# 02 · 預測下一個字

LLM 的本質出乎意料地單純：**看著前面的文字，預測下一個字**。把這件事做到極致，就成了會寫文章的模型。這堂課建立這個框架，並訓練一個最陽春的 **bigram** 模型當基線。

## 學習目標

- 理解語言模型 = 「給定前文，預測下一個 token」的機率模型
- 用 `get_batch` 切出「輸入 / 目標」訓練樣本
- 訓練一個 bigram 模型並讓它生成文字（雖然很笨）

In [1]:
text = """床前明月光，疑是地上霜。舉頭望明月，低頭思故鄉。
春眠不覺曉，處處聞啼鳥。夜來風雨聲，花落知多少。
白日依山盡，黃河入海流。欲窮千里目，更上一層樓。
紅豆生南國，春來發幾枝。願君多采擷，此物最相思。
空山不見人，但聞人語響。返景入深林，復照青苔上。
千山鳥飛絕，萬徑人蹤滅。孤舟蓑笠翁，獨釣寒江雪。
松下問童子，言師采藥去。只在此山中，雲深不知處。
人閒桂花落，夜靜春山空。月出驚山鳥，時鳴春澗中。
君自故鄉來，應知故鄉事。來日綺窗前，寒梅著花未。
獨坐幽篁裡，彈琴復長嘯。深林人不知，明月來相照。
山中相送罷，日暮掩柴扉。春草明年綠，王孫歸不歸。
功蓋三分國，名成八陣圖。江流石不轉，遺恨失吞吳。
前不見古人，後不見來者。念天地之悠悠，獨愴然而涕下。
葡萄美酒夜光杯，欲飲琵琶馬上催。醉臥沙場君莫笑，古來征戰幾人回。
秦時明月漢時關，萬里長征人未還。但使龍城飛將在，不教胡馬度陰山。
朝辭白帝彩雲間，千里江陵一日還。兩岸猿聲啼不住，輕舟已過萬重山。
故人西辭黃鶴樓，煙花三月下揚州。孤帆遠影碧空盡，唯見長江天際流。
月落烏啼霜滿天，江楓漁火對愁眠。姑蘇城外寒山寺，夜半鐘聲到客船。
獨在異鄉為異客，每逢佳節倍思親。遙知兄弟登高處，遍插茱萸少一人。
日照香爐生紫煙，遙看瀑布掛前川。飛流直下三千尺，疑是銀河落九天。
國破山河在，城春草木深。感時花濺淚，恨別鳥驚心。
岐王宅裡尋常見，崔九堂前幾度聞。正是江南好風景，落花時節又逢君。
渭城朝雨浥輕塵，客舍青青柳色新。勸君更盡一杯酒，西出陽關無故人。
清明時節雨紛紛，路上行人欲斷魂。借問酒家何處有，牧童遙指杏花村。
千里黃雲白日曛，北風吹雁雪紛紛。莫愁前路無知己，天下誰人不識君。
"""
import torch

# 字元級詞表：每個不同的字 = 一個 token
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
print(f"語料長度 {len(text)} 字，詞表大小 {vocab_size}")

語料長度 715 字，詞表大小 328


## 1. 訓練樣本長什麼樣？

每個位置的「目標」就是它的**下一個字**。輸入 `data[i:i+T]`、目標 `data[i+1:i+T+1]`——整段往右挪一格。

In [2]:
import torch

block_size = 8   # 一次看幾個字
def get_batch(batch_size=32):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])   # 往右挪一格
    return x, y

xb, yb = get_batch(1)
print("輸入:", decode(xb[0].tolist()))
print("目標:", decode(yb[0].tolist()), " ← 每個位置要預測的『下一個字』")

輸入: 孤舟蓑笠翁，獨釣
目標: 舟蓑笠翁，獨釣寒  ← 每個位置要預測的『下一個字』


## 2. 最陽春的模型：bigram

bigram 只看「當下這個字」來猜下一個字——用一張 `vocab × vocab` 的查表（`nn.Embedding`）。它沒有上下文概念，所以很笨，但足以示範整個**訓練 → 生成**流程。

In [3]:
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(0)

class Bigram(nn.Module):
    def __init__(self, V):
        super().__init__()
        self.table = nn.Embedding(V, V)   # table[i] = 看到字 i 後，下一個字的分數

    def forward(self, idx, targets=None):
        logits = self.table(idx)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n):
        for _ in range(n):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = Bigram(vocab_size)
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
for step in range(3000):
    xb, yb = get_batch()
    _, loss = model(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"訓練後 loss：{loss.item():.3f}")

訓練後 loss：1.152


In [4]:
# 從「春」開始，生成 60 個字
start = torch.tensor([[stoi["春"]]])
out = model.generate(start, 60)[0].tolist()
print(decode(out))

春眠。
空。
白帝彩半鐘聲，黃雲白日照。正為異客船。春草木深林人，夜來，獨釣寒江流盡，遍插茱萸少。
岐王孫歸不見來，時花三


結果像「有點中文味、但不通順」的字串——這完全合理：bigram 只看前一個字，記不住上下文。但**訓練 → 生成**的骨架已經完整。要讓它變聰明，模型得學會「往前看更多字」——這正是**自注意力**要解決的問題。

## 小結

- 語言模型 = 給定前文、預測下一個 token 的機率模型。
- 訓練樣本：目標是輸入「往右挪一格」。
- bigram 只看前一個字，生成必然不通順——需要上下文。

## 練習

1. 把 `start` 換成別的字開頭，看生成有何不同。
2. 印出 `model.table.weight` 中「春」那一列分數最高的幾個字——它學到「春」後面常接什麼了嗎？

下一課，認識讓模型「看見上下文」的關鍵機制：**自注意力**。